# Filter into separete unit

In [2]:
import pandas as pd
import os

# Load the Excel file
df = pd.read_excel('conv2d_0_filters.xlsx')

# Create the output folder named '64'
output_folder = '64'
os.makedirs(output_folder, exist_ok=True)

# Iterate over each column and save to a separate Excel file
for i, column in enumerate(df.columns):
    file_name = f'filter_{i+1}.xlsx'  # file names: filter_1.xlsx, filter_2.xlsx, ...
    file_path = os.path.join(output_folder, file_name)

    # Create a DataFrame with just that column
    single_col_df = df[[column]]

    # Save to Excel file
    single_col_df.to_excel(file_path, index=False)

print("Done: All 64 Excel files saved inside the '64' folder.")

Done: All 64 Excel files saved inside the '64' folder.


# Binary Conversion

In [2]:
# FILTERS
import pandas as pd
import numpy as np
import os

# Define constants
dataWidth = 16  # Total width in bits
intWidth = 4    # Integer width in bits
fracWidth = dataWidth - intWidth  # Fractional width
outputPath = "./"  # Path to save the output files
headerPath = "./"

# DtoB function for two's complement conversion
def DtoB(num, dataWidth, fracBits):  
    if num >= 0:
        num = num * (2**fracBits)
        num = int(num)
        d = num
    else:
        num = -num
        num = num * (2**fracBits)  # Number of fractional bits
        num = int(num)
        if num == 0:
            d = 0
        else:
            d = 2**dataWidth - num
    return d

# Convert the values in the dataframe to binary and save to Excel
def convert_dataframe_to_binary(df, dataWidth, fracWidth):
    binary_df = df.applymap(lambda x: format(DtoB(x, dataWidth, fracWidth), f'0{dataWidth}b'))  # Apply DtoB to all elements
    return binary_df

filter_files = [
    "all_biases.xlsx"
]

# Convert each file and save the binary data to a new Excel file
def convert_and_save_filters(filter_files):
    for filter_file in filter_files:
        # Read the filter values from the Excel file
        df = pd.read_excel(filter_file)
        
        # Convert the filter values to binary
        binary_df = convert_dataframe_to_binary(df, dataWidth, fracWidth)
        
        # Modify the output file name
        output_file = os.path.join(outputPath, f"binary_{filter_file}")
        
        # Save the binary DataFrame to a new Excel file
        binary_df.to_excel(output_file, index=False)
        print(f"Converted values saved to {output_file}")

# Convert and save all filters
convert_and_save_filters(filter_files)


In [7]:
# BIAS
import pandas as pd

# Fixed-point parameters
dataWidth = 16
intWidth = 4
fracWidth = dataWidth - intWidth

# Function to convert decimal to 4.12 fixed-point two's complement binary
def DtoB(num, dataWidth, fracBits):  
    if num >= 0:
        num = int(num * (2**fracBits))
        d = num
    else:
        num = -int(abs(num) * (2**fracBits))
        d = (1 << dataWidth) + num  # Two's complement
    return d

# Read the biases file
df = pd.read_excel("all_biases.xlsx")

# Convert third column to binary and store in new column
df["Binary_4.12"] = df.iloc[:, 2].apply(lambda x: format(DtoB(x, dataWidth, fracWidth), f'0{dataWidth}b'))

# Save the full DataFrame with the new column
df.to_excel("binary_all_biases.xlsx", index=False)
print("Converted and saved to binary_all_biases.xlsx")


Converted and saved to binary_all_biases.xlsx


In [6]:
def fixed_point_4_12_to_decimal(binary_str):
    """
    Converts a 4.12 fixed-point binary string to a signed decimal value.
    Assumes input is a 16-bit string: 4 bits integer + 12 bits fraction.
    """
    if len(binary_str) != 16:
        raise ValueError("Input must be a 16-bit binary string.")

    # Convert the full binary string to an integer
    raw = int(binary_str, 2)

    # Check for sign bit (first bit is 1 if negative)
    if binary_str[0] == '1':
        raw -= 1 << 16  # Two's complement correction

    # Divide by 2^12 to account for 12 fractional bits
    return raw / (1 << 12)


# Example usage:
binary_input = "1111111111110110"  # You can change this
decimal_value = fixed_point_4_12_to_decimal(binary_input)
print(f"Binary: {binary_input} → Decimal: {decimal_value}")
print(8*16)

Binary: 1111111111110110 → Decimal: -0.00244140625
128


# After binary

In [27]:
# FILTER
import pandas as pd

def save_filters_to_mif(file_name):
    # Load the Excel file and skip the first row
    excel_data = pd.read_excel(file_name, header=None, skiprows=1)
    
    # Iterate over each column (filter)
    for i, column in enumerate(excel_data.columns):
        filter_data = excel_data[column]
        
        # Prepare MIF content with only the binary values
        mif_content = ""
        for value in filter_data:
            # Convert values to binary (assume values are 0 or 1)
            binary_value = str(int(value))
            mif_content += f"{binary_value}\n"
        
        # Write to a MIF file
        mif_file_name = f"f_8_{i}.mif"
        with open(mif_file_name, 'w') as mif_file:
            mif_file.write(mif_content)
        print(f"Generated {mif_file_name}")

# Usage
save_filters_to_mif("binary_conv2d_7_filters.xlsx")


Generated f_8_0.mif
Generated f_8_1.mif
Generated f_8_2.mif
Generated f_8_3.mif
Generated f_8_4.mif
Generated f_8_5.mif
Generated f_8_6.mif
Generated f_8_7.mif


In [13]:
import pandas as pd
import os

# Load Excel file
df = pd.read_excel("binary_all_biases.xlsx")

# Output folder
output_dir = "separate_bias_files"
os.makedirs(output_dir, exist_ok=True)

# Process each row
for idx, row in df.iterrows():
    try:
        col1 = str(row[0]).strip()  # Layer name or ID
        col2 = str(row[1]).strip()  # Filter number or ID
        binary_value = str(row[3]).strip()  # Binary string (should be 16-bit)

        # File name: merge col1 and col2
        filename = f"{col1}_{col2}.mif"
        filepath = os.path.join(output_dir, filename)

        # Write binary value to file (just one line)
        with open(filepath, 'w') as f:
            f.write(binary_value + "\n")

        print(f"Saved: {filename}")

    except Exception as e:
        print(f"Error at row {idx}: {e}")


Saved: conv2d_1.mif
Saved: conv2d_2.mif
Saved: conv2d_3.mif
Saved: conv2d_4.mif
Saved: conv2d_5.mif
Saved: conv2d_6.mif
Saved: conv2d_7.mif
Saved: conv2d_8.mif
Saved: conv2d_9.mif
Saved: conv2d_10.mif
Saved: conv2d_11.mif
Saved: conv2d_12.mif
Saved: conv2d_13.mif
Saved: conv2d_14.mif
Saved: conv2d_15.mif
Saved: conv2d_16.mif
Saved: conv2d_17.mif
Saved: conv2d_18.mif
Saved: conv2d_19.mif
Saved: conv2d_20.mif
Saved: conv2d_21.mif
Saved: conv2d_22.mif
Saved: conv2d_23.mif
Saved: conv2d_24.mif
Saved: conv2d_25.mif
Saved: conv2d_26.mif
Saved: conv2d_27.mif
Saved: conv2d_28.mif
Saved: conv2d_29.mif
Saved: conv2d_30.mif
Saved: conv2d_31.mif
Saved: conv2d_32.mif
Saved: conv2d_33.mif
Saved: conv2d_34.mif
Saved: conv2d_35.mif
Saved: conv2d_36.mif
Saved: conv2d_37.mif
Saved: conv2d_38.mif
Saved: conv2d_39.mif
Saved: conv2d_40.mif
Saved: conv2d_41.mif
Saved: conv2d_42.mif
Saved: conv2d_43.mif
Saved: conv2d_44.mif
Saved: conv2d_45.mif
Saved: conv2d_46.mif
Saved: conv2d_47.mif
Saved: conv2d_48.mif
S

C:\Users\debab\AppData\Local\Temp\ipykernel_17824\2887816595.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  col1 = str(row[0]).strip()  # Layer name or ID
C:\Users\debab\AppData\Local\Temp\ipykernel_17824\2887816595.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  col2 = str(row[1]).strip()  # Filter number or ID
C:\Users\debab\AppData\Local\Temp\ipykernel_17824\2887816595.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`